# Structure classes from RH profiles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/huizhml/global-vegetation-structure/blob/master/notebooks/04_structure_classes.ipynb)

Each pixel's RH metrics describe how its vegetation is stacked. Clustering those profiles gives classes of vertical structure — a first, unsupervised map of *how* the vegetation is built, not only how tall it is.

The choices below (which RHs, how many classes) are illustrative; try others.

In [ ]:
%pip install -q rasterio mgrs

In [ ]:
import os
from concurrent.futures import ThreadPoolExecutor

import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.warp import transform
from rasterio.windows import from_bounds

# Reading over HTTP: skip directory listings, merge neighbouring byte ranges.
os.environ["GDAL_DISABLE_READDIR_ON_OPEN"] = "EMPTY_DIR"
os.environ["GDAL_HTTP_MERGE_CONSECUTIVE_RANGES"] = "YES"

STORE = "https://data.source.coop/geoai-ucph/gvsm"
YEAR = 2020
NODATA = 32767


def tile_url(tile, rh, year=YEAR):
    """10 m file: one per Sentinel-2 (MGRS) tile and RH. Q1 = the median prediction."""
    return f"{STORE}/{year}/{tile}/RH{rh}_Q1.tif"


def mosaic_url(rh, year=YEAR):
    """~1 km global overview: one file per RH."""
    return f"{STORE}/mosaics/{year}/RH{rh}_Q1.tif"


def read_box(url, lon, lat, half_m):
    """Read a square of 2*half_m metres around (lon, lat) from a 10 m tile, in metres.

    Only the bytes of that window are fetched: the files are Cloud-Optimized GeoTIFFs.
    Values are stored as decimetres with 32767 as nodata.
    """
    with rasterio.open(url) as src:
        xs, ys = transform("EPSG:4326", src.crs, [lon], [lat])
        x, y = xs[0], ys[0]
        win = from_bounds(x - half_m, y - half_m, x + half_m, y + half_m, src.transform)
        data = src.read(1, window=win.round_offsets().round_lengths()).astype("float32")
    data[data == NODATA] = np.nan
    return data / 10

In [ ]:
lon, lat = 9.312, 49.937  # Spessart, Germany
TILE = "32UNA"
FEATURE_RHS = list(range(10, 101, 10))  # RH10, RH20, ..., RH100

# A 3 km x 3 km window (300 x 300 pixels) of each RH, read in parallel.
with ThreadPoolExecutor(max_workers=10) as pool:
    layers = list(pool.map(lambda rh: read_box(tile_url(TILE, rh), lon, lat, half_m=1500), FEATURE_RHS))
stack = np.stack(layers, axis=-1)  # rows x cols x features
valid = np.all(np.isfinite(stack), axis=-1)
X = stack[valid]
print(stack.shape, "->", X.shape[0], "valid pixels")

In [ ]:
from sklearn.cluster import KMeans

K = 5
km = KMeans(n_clusters=K, n_init=10, random_state=0).fit(X)

# Order the classes by their mean canopy height, so class 0 is the lowest.
order = np.argsort(km.cluster_centers_[:, -1])
rank = np.empty(K, int)
rank[order] = np.arange(K)

classes = np.full(valid.shape, np.nan)
classes[valid] = rank[km.labels_]

In [ ]:
cmap = plt.get_cmap("viridis", K)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
im = axes[0].imshow(classes, cmap=cmap, vmin=-0.5, vmax=K - 0.5)
axes[0].set_title("Structure classes")
axes[0].axis("off")
fig.colorbar(im, ax=axes[0], ticks=range(K), shrink=0.8)

for k in range(K):
    centre = km.cluster_centers_[order[k]]
    axes[1].plot(FEATURE_RHS, centre, color=cmap(k), lw=2, label=f"class {k}")
axes[1].set_xlabel("RH n (% of returned energy)")
axes[1].set_ylabel("height (m)")
axes[1].set_title("Mean RH profile of each class")
axes[1].legend()
plt.show()